### Purpose of this exercise:

- Get a feeling for how it feels to train a model
- Evaluate the model predictions
- Explore different metrics of model evaluation
- Develop your own model
- Compare model performances against the rest of the class

## Step 1: Getting set up

The first step is to import all our relevent functions and load the data.

Here we will use standard python libraries, as well as a number of helper functions borrowed from FLUXtrapolation which make the training and evaluation easier.

**Make sure you have already downloaded the site data (see the if you haven't downloaded yet [README](README.md))**

In [ ]:
import numpy as np
import matplotlib.pyplot as pl

from dataloader import load_data, get_data_split, save_predictions, load_predictions
from utils.eval_utils import compute_and_save_metrics, save_best_params
from utils.aggregation import AGGREGATIONS as aggregations
from eval import load_all_metrics, display_names, class_leader
from utils.plots import plot_cdf, create_html_leaderboard, MODEL_COLORS

Here we set some basic setting used throughout the exercise:

- `setting` corresponds to which site we will hold out as our test set.
    - `'TA40'` leaves out the sites with the highest average annual temperatur, this is the hardest test set
    - `'spatial-easy40'` leaves out a random set of sites, which is slightly easier as there is less extrapolation
    - `'time-split'` does not leave out whole sites, but rather leaves out the last years from long term sites. This is by far the easist for the model.
- `target` is the flux we will try to model
- `strategy` is how to summarize the evaluation scores across the holdout sites
    - `'mean'` takes the mean (e.g. the mean RMSE across hold out sites)
    - `'max'` takes the 90th quantile (e.g. the mean RMSE across hold out sites)
    - `'mean'` takes the mean (e.g. the mean RMSE across hold out sites
- `yourname` is simply your name, so we can identify your models

In [ ]:
setting = 'spatial-easy40' # 'time-split', 'spatial-easy40', 'TA40'
target = "GPP" # 'GPP', 'NEE', 'ET'
strategy = "mean" # 'mean', 'max', 'discrepancy'
yourname = "" # Make it unique!

assert len(yourname) > 0, "You need to put your name!"

Here we load the data and create our training, validation, and test splits.

*Note: if you change `setting` to a different hold out set, you will need to reload the data for it to take effect*

In [ ]:
df = load_data("data")

train, val, test, col_names, _ = get_data_split(
    df,
    setting,
    remove_missing_target=True,
    path="data",
    return_colnames=True,
    standardize=False, #model_name in ['robust-lr', 'ridge', 'mlp', 'gdro', 'coral', 'mmd'],
    astorch=False, # model_name in ['mlp', 'gdro', 'coral', 'mmd']
)

xtrain, ytrain, envs_train = train
xval, yval, envs_val = val
xtest = test[0]

The data we are using is a simplified version of what we use in FLUXCOM, including the key meteorological, remote sensing data, as well as PFT categories.

Run the next cell to get an overview of the data.

A full description of the data can be found [here](https://arxiv.org/html/2605.19812v1#A1).

In [ ]:
df

## Step 2: Your first model

To start, we will run a simple linear regression model through the full training, prediction, and evaluation cycle. You can then use this linear regression as a baseline for further models.

First we initiate the model and give it a name,

In [ ]:
from sklearn.linear_model import LinearRegression
lr_model = LinearRegression()
model_name = f"{yourname}-lnreg"

Next, we simply fit the model. As this is a linear regression, we do not need a separate validation set, so we concatinate the validation and training into a single dataset. Many other models will use the validation set to tell the model when to stop.

In [ ]:
lr_model.fit(
    X=np.concat([xval, xtrain]),
    y=np.concat([yval, ytrain])
)

Now that the model is fit, we will predict on the test dataset. The `save_predictions` function will save the predictions in case you want to look at them again later.

In [ ]:
ypred = lr_model.predict(xtest)
preds_df = save_predictions(test, ypred, setting, target, model_name, val_strategy=strategy)
preds_df

Now let's explore the outputs to see how the predictions look. We will start by looking at the predictions for a single site. Run the cell below to see a list of sites we can look at from the test dataset.

In [ ]:
preds_df.site_id.unique().tolist()

In [ ]:
site = "" # <= Copy the site code here

assert site in preds_df.site_id.unique().tolist(), f"You need to pick a site from the list above before you can continue, so far you chose: {site}"

site_df = preds_df[preds_df.site_id==site] # subselect the data from only the site
site_df['time'] = site_df["time"].astype("datetime64[s]") # convert the time column into a datetime object

print(f"This site has data from {site_df.time.min()} to {site_df.time.max()}")

Now we will take a look at what the data looks like. As hourly data is hard to see when it is all bunched together, pick a time window of about a month to look at. Note that it needs to be within the time range that the site has data (see above for the time range for this site).

In [ ]:
start_date = "2015-05-01"
end_date = "2015-05-31"

subset = site_df.set_index("time")[start_date:end_date]

subset["y_true"].plot(figsize=(12,4), label="Observations")
subset["y_pred"].plot(label="Predictions")

pl.legend()

Note that the linear regression model doesn't quite seem to get the diurnal patters fully correct.

Let's look at another timescale to see if it is any better. There are already aggregation functions you can use, here is an example to look at daily values for one year. Be sure to select an appropriate time range!

In [ ]:
start_date = "2015-01-01"
end_date = "2015-12-31"

subset = aggregations["daily"](site_df).set_index("time")[start_date:end_date]

subset["y_true"].plot(figsize=(12,4), label="Observations")
subset["y_pred"].plot(label="Predictions")

pl.legend()

Now try making some more plots using different scales, as well as looking at different sites.

What aspects seem to be working?
Where does the model struggle?
Are there any aspects that seem odd?

Run the cell below to see the available aggregation functions.

In [ ]:
aggregations.keys()

## Step 3: Compute metrics across all sites

Now that you have looked at a few different sites and aggregations, it is now time to look at the big picture.

Below you will compute all the metrics for all scales to produce a large dataset describing how well your model has worked. These will also be saved in case you would like to look at them later. You can find all saved metrics in the `results/metrics/` folder in this repository.

Take a look at the table produced, particularly the column names corresponding to evaluation metrics. If you are unsure about any of the metrics, find the formulations in [utils/eval_utils.py](utils/eval_utils.py).

In [ ]:
metrics = compute_and_save_metrics(preds_df, setting, target, model_name, val_strategy=strategy)
metrics

Pick a metric from the column list and the cell below will give an quick overview of the performance based on that metric.

In [ ]:
metric = "" # <= put a metric here!

assert metric in metrics.columns, f"Pick a metric from the columns above, you put '{metric}' which is not a valid metric" 



scales = ['hourly', 'daily', 'weekly', 'monthly', 'seasonal', 'anom', 'iav']

fig, axs = pl.subplots(nrows=1, ncols=2, figsize=(12, 6))

# plot violin plot
ax = axs[0]
ax.violinplot(
    [metrics[metrics.scale==scale][metric] for scale in scales],
    showmeans=False,
    showmedians=True,
    showextrema=False,
)
ax.set_title(target, loc="left", weight="bold")
ax.yaxis.grid(True)
ax.set_xticks([y + 1 for y in range(len(scales))],
              labels=scales)
ax.set_ylabel(metric)


scale="spatial"
ax = axs[-1]
spatial_df = aggregations["spatial"](preds_df)
ax.scatter(
    spatial_df["y_pred"],
    spatial_df["y_true"],
    s=20
)
drange = [0, 10]
ax.plot(drange, drange, "k--", lw=1)
ax.set_xlim(drange)
ax.set_ylim(drange)
ax.set_xlabel(f"Predicted {target}")
ax.set_ylabel(f"Observed {target}")
ax.set_title(scale)

Take a look at some different metrics for the linear regression. Then we can move on to more advanced models.

## Step 4: Bigger, Better?, Slower, Stronger

Now with the linear regression as our baseline, lets try to train a model with a bit more flexibility. Here we will run the same analysis, but only switchin out the model we use.

In this case, we will use a boosted regression trees model from [XGBoost](https://xgboost.readthedocs.io/en/release_3.2.0/tutorials/model.html).

#### Model set up

One key difference of the more advanced models are *hyperparameters*, which aonfiguration setting or variable that you manually define before training a model. These can have relatively large effects on the model final model performance, so they are often tuned by trying many different combinations of hyperparameters to find the best combinations. However, we will just run a single set in this example. Feel free to come back and adjust them later.

In [ ]:
from xgboost import XGBRegressor
xgb_model = XGBRegressor(
     early_stopping_rounds = 10,
     n_jobs                = 1, # Feel free to increase this if your computer has multiple processors. This will only have the effect of speeding up the trainging.
     learning_rate         = 0.05,
     colsample_bynode      = 1,
     objective             = 'reg:squarederror',
     subsample             = 0.5,
     tree_method           = 'hist',
     min_child_weight      = 5,
)

model_name = f"{yourname}-XGBoost"

Once we set up the new model, we will again fit to the trianing datset.
    
*Note that the validation set is now being used to determine when to stop training the model, which is different from what we did for the linear regression.*

Training can take some time, but XGBoost is pretty fast.

In [ ]:
xgb_model.fit(
    X=xtrain,
    y=ytrain,
    eval_set = [(xval, yval)],
    verbose = 10,
)

Just as last time, we predict on the test dataset and save the predictions.

In [ ]:
ypred = xgb_model.predict(xtest)
preds_df = save_predictions(test, ypred, setting, target, model_name, val_strategy=strategy)

Now that you have a new set of predictions, create some of the interesting plots from before.

Does the new model seem to perform better than linear regresssion?
Are there any aspects it does not perform better?
What are the aspects that the model still struggles with?

In [ ]:
site_df = preds_df[preds_df.site_id==site] # subselect the data from only the site
site_df['time'] = site_df["time"].astype("datetime64[s]") # convert the time column into a datetime object

In [ ]:
start_date = "2015-05-01"
end_date = "2015-05-31"

subset = site_df.set_index("time")[start_date:end_date]

subset["y_true"].plot(figsize=(12,4), label="Observations")
subset["y_pred"].plot(label="Predictions")

pl.legend()

In [ ]:
start_date = "2015-01-01"
end_date = "2015-12-31"

subset = aggregations["daily"](site_df).set_index("time")[start_date:end_date]

subset["y_true"].plot(figsize=(12,4), label="Observations")
subset["y_pred"].plot(label="Predictions")

pl.legend()

The following cell will load the predictions from the linear regression. Make a plot that compares both models to the observations.

In [ ]:
lr_pred_df = load_predictions(setting, target, f"{yourname}-lnreg", val_strategy=strategy)

Finally, we will compute and save the metrics just as before.

In [ ]:
metrics = compute_and_save_metrics(preds_df, setting, target, model_name, val_strategy=strategy)

In [ ]:
scales = ['hourly', 'daily', 'weekly', 'monthly', 'seasonal', 'anom', 'iav']
metric = "rmse"

fig, axs = pl.subplots(nrows=1, ncols=2, figsize=(12, 6))

# plot violin plot
ax = axs[0]
ax.violinplot(
    [metrics[metrics.scale==scale][metric] for scale in scales],
    showmeans=False,
    showmedians=True,
    showextrema=False,
)
ax.set_title(target, loc="left", weight="bold")
ax.yaxis.grid(True)
ax.set_xticks([y + 1 for y in range(len(scales))],
              labels=scales)
ax.set_ylabel(metric)


scale="spatial"
ax = axs[-1]
spatial_df = aggregations["spatial"](preds_df)
ax.scatter(
    spatial_df["y_pred"],
    spatial_df["y_true"],
    s=20
)
drange = [0, 10]
ax.plot(drange, drange, "k--", lw=1)
ax.set_xlim(drange)
ax.set_ylim(drange)
ax.set_xlabel(f"Predicted {target}")
ax.set_ylabel(f"Observed {target}")
ax.set_title(scale)

Compare the metrics between the linear regression and XGBoost.

Are there clear differences?
Is one model better in all categories?

**Note:** you can load multiple model metrics into a single dataframe with:

In [ ]:
all_metrics = load_all_metrics(
    settings=setting,
    targets=target,
    models=[f"{yourname}-XGBoost", f"{yourname}-lnreg"],
    val_strategy=strategy
)
all_metrics

## Step 5: Choose your own adventure

Now that you have mastered a couple models, now it is time to try something new and unexpected. The goal is to develop a new type of model. Here are a couple options to get you started (from easy to ambitious):

- Option 0: Improve either the XGBoost or Linear Regression models from above
- Option 1: Use a different method from SciKit-Learn
- Option 2: Create neural network method using pytorch
- Option 3: Create something completely different

**Note:** Due to the time we have, best would be to pick one. You can always do the others later.

### Option 0

No example needed, just copy some code from above and change whatever you think will improve the model. Remember to give it a unique name.

### Option 1

Because SciKit-Learn is already installed, and all of their model implementations have been standardized, you can relatively easily implement them into the same framework.

There is a lot of information [on the SciKit-Learn website](https://scikit-learn.org/stable/supervised_learning.html), just note that some methods are easier to implement than others. And also some methods can take a long time to fit, especially if you are running on your own laptop.

Feel free to ask for a recommendation if you are having trouble.

In [ ]:
from sklearn import ...  # <= import model here and initialize
my_model = ...

model_name = f"{yourname}-PutNewModelNameHereSomethingUnique"

In [ ]:
my_model.fit(...

In [ ]:
ypred = my_model.predict(xtest)
preds_df = save_predictions(test, ypred, setting, target, model_name, val_strategy=strategy)

metrics = compute_and_save_metrics(preds_df, setting, target, model_name, val_strategy=strategy)

### Option 2

If you are familiar with pytorch, the framework here is already set up for directly integrating into the workflow with just a couple small changes.

First, the data needs to be loaded slightly differently, as the train, val and test objects need to be tensors rather than numpy arrays.

In [ ]:
# df = load_data("data")

# train, val, test, col_names, _ = get_data_split(
#     df,
#     setting,
#     remove_missing_target=True,
#     path="data",
#     return_colnames=True,
#     standardize=True,
#     astorch=True,
# )

# xtrain, ytrain, envs_train = train
# xval, yval, envs_val = val
# xtest = test[0]

Once the data is set up, the easiest way to implement a new model would be to copy and modify the multilayer perceptron example, you can find it in this repository at [models/mlp.py](models/mlp.py). Note that the training time takes a while on my laptop, so it might need to run for an hour or so.

Otherwise the implementation is exactly the same.

In [ ]:
from models.mlp import MLP
xgb_model = MLP(...

model_name = f"{yourname}-MLP"

In [ ]:
my_model.fit(...

In [ ]:
ypred = my_model.predict(xtest)
preds_df = save_predictions(test, ypred, setting, target, model_name, val_strategy=strategy)

metrics = compute_and_save_metrics(preds_df, setting, target, model_name, val_strategy=strategy)

### Option 3

The framework is actually agnostic to what your model does, so you can really run wild. As long as the model has `fit` and `predict` functions, then everything else should work.

**Note:** that you could even develop a simple mechanistic model. If you need the data names, you can identify each variable from `col_names`. For example:

```py
xtrain[:,col_names.index("TA")]
```

In [ ]:
class MyModel:
    def __init__(self, param1=1.0):
        self.param1 = param1

    def fit(self, X, y, eval_set=None, envs=None):
        """
        X, y   : np.ndarray (or torch.Tensor if you opt into tensor mode — see Step 4)
        eval_set: optional [(X_val, y_val)] for early stopping
        envs   : optional array of site labels, for group-aware methods
        """
        ...
        return self

    def predict(self, X):
        """Returns np.ndarray of shape (n_samples,)."""
        ...

## Step 6: Intercompare across all models

Now we have model output stacking up, we need to intercompare them. First we need to load all the model metrics we have produced so far.

In [ ]:
all_metrics = load_all_metrics(
    settings=setting,
    targets=target,
    val_strategy=strategy
)

In [ ]:
all_metrics.model.unique().tolist()

The world always needs color, and it makes it easier to identify different models, so assign a color to each model. You can use any [html color name](https://www.w3schools.com/tags/ref_colornames.asp), then update for each model:

In [ ]:
MODEL_COLORS[f"{yourname}-lnreg"] = "CornflowerBlue"
MODEL_COLORS[f"{yourname}-XGBoost"] = "IndianRed"

One useful plot to intercompre different models is a cumulative distribution plot of a metric. To interpret these plots, follow the metric on the x axis, where the corresponding point on the y axis indicates the fraction of sites that have that metric or better. This gives an overview of the full distribution of site performance. Note that the horizontal line at 0.5 corresponds to the median of the metric.

In [ ]:
fig, ax = pl.subplots(
    nrows=1,
    ncols=1,
    figsize=(6,6)
)

plot_cdf(all_metrics, target, metric, scale="hourly", setting=setting, ax=ax, xmax=None,
             setting_name=None, linestyle='-', linewidth=2)

An even more condensed version is a leader board. Here the information is condensed into a single table. Here, the color colors indicate relative performance within each column, with darker green denoting the best value and fading to white at 1.2 times the best value. The "Skill score" reports summary score relative to linear regression.

In [ ]:
from IPython.display import display, HTML

display(HTML(
    create_html_leaderboard(all_metrics, target, metric='rmse',
                        aggfunc='median',
                        filename=f'results/plots/leaderboard_{target}.html')
))

Once you have some models you are satisfied with, you can upload your metrics table to compare against the rest of the class. Simple upload the corresponding metrics file, which are automatically save at `results/metrics/`, into [this shared folder](https://nextcloud.bgc-jena.mpg.de/s/RCr73c8C8JmasNN).

Once uploaded, you can see how you compare to all the other models with the following code:

In [ ]:
class_leader(target, metric='rmse', aggfunc='median')